# Рекомендательная система фильмов

Рекомендательная система - это программа, которая угадывает, какой человек захочет посмотреть следующим,
и предлагает ему несколько вариантов.

 четыре способа рекомендаций:

1. **Популярное** - показывать то, что понравилось всем больше всего.
2. **Похожие зрители** - что советуют люди с таким же вкусом, как у тебя.
3. **Похожие фильмы** - что похоже на то, что тебе уже понравилось.
4. **По содержанию** - сравнение названий и жанров, работает даже без единой оценки.

Эти же четыре способа потом работают в API и в приложении, а сама логика лежит в одном файле -
`app/recommender.py`.

## Откуда данные

Фильмы и оценки взяты из открытого набора MovieLens - его собирает исследовательская группа Миннесотского университета,
и это стандартный материал для таких работ. Архив скачали один раз, он лежит в папке `data/`, поэтому при
запуске тетради интернет не нужен и ни с какого сайта ничего не вытаскивается.

| Файл | Записей | Что внутри |
|---|---|---|
| `movies.csv` | 9 742 | номер фильма, название, жанры (перечислены через `\|`) |
| `ratings.csv` | 100 836 | кто оценил, какой фильм и какая оценка (от 0.5 до 5.0) |

Оценки ставят 610 человек, и каждый оценил не меньше 20 фильмов — на таком материале уже есть с чем
работать. Колонка с датой оценки не используется.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 60)

DATA = Path("data")  # тетрадь лежит в корне проекта, рядом с папкой data

**Загрузка данных**

In [2]:
movies = pd.read_csv(DATA / "movies.csv")
ratings = pd.read_csv(DATA / "ratings.csv")
print("фильмов:", len(movies), "| оценок:", len(ratings), "| пользователей:", ratings["userId"].nunique())

фильмов: 9742 | оценок: 100836 | пользователей: 610


In [3]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


**Приведение таблиц в порядок**

Из названия убраны год в скобках, жанры стали читаться через запятую, колонка с датой удалена.
Все алгоритмы дальше работают с этой чистой таблицей - так же, как и API.

In [5]:
movies = movies.copy()
movies["title"] = movies["title"].str.replace(r"\s*\(\d{4}\)\s*$", "", regex=True)
ratings = ratings.drop(columns=["timestamp"])

info = movies.merge(
    ratings.groupby("movieId")["rating"].agg(average="mean", count="count"),
    left_on="movieId",
    right_index=True,
    how="left",
).fillna({"average": 0, "count": 0})
info["count"] = info["count"].astype(int)
info["genres_list"] = info["genres"].str.replace("|", ", ", regex=False)
info.head()

,movieId,title,genres,average,count,genres_list
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,3.920930,215,"Adventure, Animation, Children, Comedy, Fantasy"
1,2,Jumanji,Adventure|Children|Fantasy,3.431818,110,"Adventure, Children, Fantasy"
2,3,Grumpier Old Men,Comedy|Romance,3.259615,52,"Comedy, Romance"
3,4,Waiting to Exhale,Comedy|Drama|Romance,2.357143,7,"Comedy, Drama, Romance"
4,5,Father of the Bride Part II,Comedy,3.071429,49,Comedy


In [6]:
print("Оценки по всему датасету:")
print(ratings["rating"].describe().round(2))
print()
print("Сколько фильмов оценил пользователь (медиана):", int(ratings["userId"].value_counts().median()))
print("Фильмов без оценок:", int((info["count"] == 0).sum()), "- их алгоритмы не советуют")

Оценки по всему датасету:
count    100836.00
mean          3.50
std           1.04
min           0.50
25%           3.00
50%           3.50
75%           4.00
max           5.00
Name: rating, dtype: float64

Сколько фильмов оценил пользователь (медиана): 70
Фильмов без оценок: 18 — их алгоритмы не советуют


## 1. Популярные фильмы

Самый простой способ - отсортировать фильмы по средней оценке. Но тогда в топ попадёт фильм, который
оценил один человек и поставил высший балл. Поэтому рейтинг делают осторожным: если оценок мало,
результат подтягивается к средней оценке по всему датасету, а когда оценок много - считается уже
собственная оценка фильма. 

In [7]:
# Что показывает простая сортировка по средней оценке
info.sort_values("average", ascending=False).head(5)[["title", "average", "count"]]

,title,average,count
9711,Won't You Be My Neighbor?,5.0,1
48,Lamerica,5.0,2
5690,'Salem's Lot,5.0,1
7239,Meantime,5.0,1
8873,What Love Is,5.0,1


In [8]:
C = ratings["rating"].mean()  # средняя оценка по датасету
m = 20                        # чем меньше оценок, тем сильнее результат тянется к среднему

v = info["count"]
info["w_score"] = (v / (v + m)) * info["average"] + (m / (v + m)) * C

top_10 = info.sort_values("w_score", ascending=False).head(10)
top_10[["title", "genres_list", "average", "count", "w_score"]]

,title,genres_list,average,count,w_score
277,"Shawshank Redemption, The","Crime, Drama",4.429022,317,4.373980
659,"Godfather, The","Crime, Drama",4.289062,192,4.214770
2226,Fight Club,"Action, Crime, Drama, Thriller",4.272936,218,4.208114
224,Star Wars: Episode IV - A New Hope,"Action, Adventure, Sci-Fi",4.231076,251,4.177237
46,"Usual Suspects, The","Crime, Mystery, Thriller",4.237745,204,4.172014
461,Schindler's List,"Drama, War",4.225000,220,4.164713
922,"Godfather: Part II, The","Crime, Drama",4.259690,129,4.157927
257,Pulp Fiction,"Comedy, Crime, Drama, Thriller",4.197068,307,4.154529
898,Star Wars: Episode V - The Empire Strikes Back,"Action, Adventure, Sci-Fi",4.215640,211,4.153814
6710,"Dark Knight, The","Action, Crime, Drama, IMAX",4.238255,149,4.151072


В итоге в топе: «Побег из Шоушенка», «Крёстный отец», «Бойцовский клуб». Именно этот
список отдаёт адрес `GET /api/v1/movies/popular?limit=10`.

Теперь нужна таблица «кто какой фильм оценил» - на ней построены оба способа с похожими людьми и
похожими фильмами.

In [9]:
user_ids = sorted(ratings["userId"].unique())
movie_ids = info["movieId"].tolist()
user_index = {u: i for i, u in enumerate(user_ids)}
movie_index = {mid: i for i, mid in enumerate(movie_ids)}

matrix = (
    ratings.pivot(index="userId", columns="movieId", values="rating")
    .reindex(index=user_ids, columns=movie_ids)
    .fillna(0)
    .to_numpy()
)
matrix.shape

(610, 9742)

**Поправка**

Одни люди ставят 4-5 почти всему, другие редко дают больше тройки. Если сравнивать такие
оценки как есть, кажется, что всем нравятся все фильмы. Поэтому из каждой оценки вычитается среднее этого
человека: остаётся не «четвёрка», а «на сколько он щедрее обычного». После этого сравнивать людей
становится честнее.

In [10]:
n_rated = (matrix > 0).sum(axis=1)
means = matrix.sum(axis=1) / np.maximum(n_rated, 1)
centered = np.where(matrix > 0, matrix - means[:, None], 0)

titles = info["title"].to_numpy()
genres_arr = info["genres_list"].to_numpy()
averages = info["average"].to_numpy().round(2)
counts = info["count"].to_numpy()
print("средняя оценка пользователя 1:", round(means[user_index[1]], 2))

средняя оценка пользователя 1: 4.37


In [11]:
def to_table(indices, scores, score_name="Прогноз"):
    """Один формат вывода для всех алгоритмов — такой же, как в API."""
    return pd.DataFrame({
        "Фильм": titles[indices],
        "Жанры": genres_arr[indices],
        "Средняя оценка": averages[indices],
        "Оценок": counts[indices],
        score_name: np.round(scores, 2),
    })

## 2. Рекомендации по похожим людям и по похожим фильмам

Этот способ ничего не знает о содержании фильма - он смотрит только на то, что и кому понравилось.
Бывает двух видов:

- **похожие зрители** - находим людей с близким вкусом и сравниваем их оценки этого фильма;
- **похожие фильмы** - сравниваем сами фильмы по тому, кто ставил им высокие оценки, и советуем то,
  что похоже на то, что нравилось ранье.

### Способ «похожие зрители»

Берём 50 людей, чьи вкусы ближе всего к нашему (сходство считается по уже исправленным оценкам), и для
каждого фильма усредняем их оценки: чем больше похожесть человека, тем больший вес имеет его оценка.

Фильм попадает в список, только если его оценили минимум три похожих зрителя, у него вообще не меньше
пяти оценок в датасете и этот человек его ещё не смотрел.

In [12]:
def recommend_by_users(user_id, k=10, n_neighbors=50, min_support=3):
    i = user_index[user_id]
    rated = matrix[i] > 0

    sim = cosine_similarity(centered[[i]], centered)[0]
    sim[i] = 0                      # самого себя соседом не считаем
    sim[sim < 0] = 0                # отрицательное сходство не используем
    neighbors = np.argsort(sim)[-n_neighbors:]

    weights = sim[neighbors, None] * (matrix[neighbors] > 0)
    num = (weights * matrix[neighbors]).sum(axis=0)
    den = weights.sum(axis=0)
    support = (weights > 0).sum(axis=0)

    scores = np.full(len(movie_ids), -np.inf)
    valid = (den > 0) & (support >= min_support) & (counts >= 5) & ~rated
    scores[valid] = num[valid] / den[valid]

    order = [j for j in np.argsort(scores)[::-1] if np.isfinite(scores[j])][:k]
    return to_table(np.array(order), np.clip(scores[order], 0.5, 5))

In [13]:
recommend_by_users(1)

,Фильм,Жанры,Средняя оценка,Оценок,Прогноз
0,Empire of the Sun,"Action, Adventure, Drama, War",3.95,10,4.84
1,Paths of Glory,"Drama, War",4.54,12,4.71
2,Wallace & Gromit: The Best of Aardman Animation,"Adventure, Animation, Comedy",4.09,27,4.69
3,Defending Your Life,"Comedy, Drama, Fantasy, Romance",3.85,10,4.67
4,"Godfather, The","Crime, Drama",4.29,192,4.65
5,Old Boy,"Mystery, Thriller",4.09,39,4.65
6,Moon,"Drama, Mystery, Sci-Fi, Thriller",3.97,32,4.64
7,Seven Samurai (Shichinin no samurai),"Action, Adventure, Drama",4.19,48,4.63
8,"Conversation, The","Drama, Mystery",4.21,17,4.62
9,Dr. Strangelove or: How I Learned to Stop Worrying and L...,"Comedy, War",4.27,97,4.60


### Способ «похожие фильмы»

Здесь сравниваются уже фильмы - насколько одинаково люди их оценивают. Прогноз строится так: берём
обычное среднее и добавляем поправку - насколько сильно и насколько уверенно похожие фильмы отклонялись
от нормы. Благодаря этому результат остаётся в понятной шкале от 0.5 до 5.0.

У очень активного пользователя тысячи оценок, поэтому в расчёт берутся не больше 300 самых показательных - иначе таблица сравнения фильмов займёт слишком много памяти.

In [14]:
def recommend_by_items(user_id, k=10, min_support=3, max_rated=300):
    i = user_index[user_id]
    rated = matrix[i] > 0
    rated_idx = np.flatnonzero(rated)
    if len(rated_idx) > max_rated:  # берём оценки с наибольшим отклонением от среднего
        keep = np.argsort(np.abs(matrix[i][rated_idx] - means[i]))[-max_rated:]
        rated_idx = rated_idx[keep]

    sim = cosine_similarity(centered[:, rated_idx].T, centered.T)  # сравнение этих фильмов со всеми остальными
    dev = matrix[i][rated_idx] - means[i]
    num = dev @ sim
    den = np.abs(sim).sum(axis=0)
    support = (np.abs(sim) > 1e-8).sum(axis=0)

    scores = np.full(len(movie_ids), -np.inf)
    valid = (den > 0) & (support >= min_support) & (counts >= 5) & ~rated
    scores[valid] = means[i] + num[valid] / den[valid]

    order = [j for j in np.argsort(scores)[::-1] if np.isfinite(scores[j])][:k]
    return to_table(np.array(order), np.clip(scores[order], 0.5, 5))

In [15]:
recommend_by_items(1)

,Фильм,Жанры,Средняя оценка,Оценок,Прогноз
0,Garden State,"Comedy, Drama, Romance",3.71,48,4.78
1,To Kill a Mockingbird,Drama,4.12,58,4.77
2,Rain Man,Drama,3.81,96,4.77
3,"Bridge Too Far, A","Action, Drama, War",3.79,7,4.76
4,"Dreamers, The",Drama,3.73,13,4.75
5,The Artist,"Comedy, Drama, Romance",4.05,11,4.75
6,Beautiful Girls,"Comedy, Drama, Romance",3.95,10,4.75
7,Tigerland,Drama,3.56,8,4.75
8,Frost/Nixon,Drama,3.94,9,4.74
9,Superbad,Comedy,3.86,55,4.74


## 3. Рекомендации по содержанию

Похожие люди и похожие фильмы плохо работают для нового фильма или нового человека  оценок ещё нет.
Тогда помогает сравнение содержания: из названия и жанров составляется текст, он переводится в набор
чисел, и фильмы сравниваются между собой по тому, насколько совпадают их слова. Оценки здесь не нужны
вообще.

In [16]:
texts = (info["title"] + " " + info["genres"].str.replace("|", " ", regex=False)).tolist()
vectors = TfidfVectorizer().fit_transform(texts)
vectors.shape

(9742, 9187)

In [17]:
def recommend_by_content(title, k=10):
    i = np.flatnonzero(titles == title)[0]
    sim = cosine_similarity(vectors[i], vectors)[0]
    sim[i] = -1  # фильм сам себе не рекомендация
    order = np.argsort(sim)[-k:][::-1]
    return to_table(order, sim[order], "Сходство")

In [18]:
recommend_by_content("Toy Story")

,Фильм,Жанры,Средняя оценка,Оценок,Сходство
0,Toy Story 2,"Adventure, Animation, Children, Comedy, Fantasy",3.86,97,1.00
1,Toy Story 3,"Adventure, Animation, Children, Comedy, Fantasy, IMAX",4.11,55,0.93
2,"Toy, The",Comedy,2.38,4,0.67
3,Toy Soldiers,"Action, Drama",2.50,4,0.46
4,"NeverEnding Story, The","Adventure, Children, Fantasy",3.58,43,0.45
5,L.A. Story,"Comedy, Romance",3.48,24,0.44
6,"Christmas Story, A","Children, Comedy",3.97,55,0.43
7,"Wild, The","Adventure, Animation, Children, Comedy, Fantasy",2.50,1,0.42
8,"NeverEnding Story III, The","Adventure, Children, Fantasy",2.00,7,0.40
9,Home,"Adventure, Animation, Children, Comedy, Fantasy, Sci-Fi",3.25,8,0.38


 что отдаёт адрес `GET /api/v1/recommendations/content/{movie_id}`.

In [19]:
for t in top_10["title"].tolist():
    similar = recommend_by_content(t, k=3)["Фильм"].tolist()
    print(f"{t:<28} -> {', '.join(similar)}")

Shawshank Redemption, The    -> The Raid: Redemption, Redemption (Hummingbird), Undisputed III: Redemption
Godfather, The               -> Godfather: Part II, The, Godfather: Part III, The, The Godfather Trilogy: 1972-1990
Fight Club                   -> Fist Fight, X-Files: Fight the Future, The, Club Paradise
Star Wars: Episode IV - A New Hope -> Star Wars: Episode I - The Phantom Menace, Star Wars: Episode V - The Empire Strikes Back, Star Wars: Episode III - Revenge of the Sith
Usual Suspects, The          -> Z, Double, The, F/X
Schindler's List             -> Bucket List, The, Kill List, To Do List, The
Godfather: Part II, The      -> Godfather, The, Godfather: Part III, The, Hangover Part II, The
Pulp Fiction                 -> Stranger than Fiction, F/X, K-9
Star Wars: Episode V - The Empire Strikes Back -> Star Wars: Episode I - The Phantom Menace, Star Wars: Episode IV - A New Hope, Star Wars: Episode III - Revenge of the Sith
Dark Knight, The             -> Dark Knight Rises,

Одно честное ограничение: сравниваются слова, а не смысл. Поэтому к «Schindler's List» в похожие попадают
«Bucket List» и «Kill List»  совпало только слово «список». Отчасти это лечится тем, что в текст вместе
с названием входят жанры.

## 4. Персональные рекомендации

Последняя функция собирает всё вместе. Если у человека нет ни одной оценки (в списке он значится как
«Новый пользователь») или похожих зрителей не нашлось, вместо персональных рекомендаций показываются
популярные. Флаг `fallback` сообщает об этом интерфейсу, и приложение честно пишет сверху, что
предложения общие.

In [20]:
def recommend(user_id, method="user", k=10):
    if user_id not in user_index or (matrix[user_index[user_id]] > 0).sum() == 0:
        rows = top_10.head(k)
        table = pd.DataFrame({
            "Фильм": rows["title"],
            "Жанры": rows["genres_list"],
            "Средняя оценка": rows["average"].round(2),
            "Оценок": rows["count"],
            "Рейтинг": rows["w_score"].round(2),
        }).reset_index(drop=True)
        return table, True
    table = recommend_by_users(user_id, k) if method == "user" else recommend_by_items(user_id, k)
    return table, False

In [21]:
table, fallback = recommend(0, "user")  # id 0 — «Новый пользователь» без единой оценки
print("Это запасной вариант (popular)?", fallback)
table.head()

Это запасной вариант (popular)? True


,Фильм,Жанры,Средняя оценка,Оценок,Рейтинг
0,"Shawshank Redemption, The","Crime, Drama",4.43,317,4.37
1,"Godfather, The","Crime, Drama",4.29,192,4.21
2,Fight Club,"Action, Crime, Drama, Thriller",4.27,218,4.21
3,Star Wars: Episode IV - A New Hope,"Action, Adventure, Sci-Fi",4.23,251,4.18
4,"Usual Suspects, The","Crime, Mystery, Thriller",4.24,204,4.17


In [22]:
for user in (1, 42, 198):
    n = int((matrix[user_index[user]] > 0).sum())
    print(f"Пользователь {user} ({n} оценок) →", ", ".join(recommend(user)[0]["Фильм"].tolist()[:3]), "...")

Пользователь 1 (232 оценок) → Empire of the Sun, Paths of Glory, Wallace & Gromit: The Best of Aardman Animation ...


Пользователь 42 (440 оценок) → Yojimbo, Cool Hand Luke, High Noon ...


Пользователь 198 (230 оценок) → City of Lost Children, The (Cité des enfants perdus, La), Gravity, Cool Hand Luke ...


## Проверка результатов

Четыре простые проверки - они же есть в тестах `tests/test_api.py`.

In [23]:
u = recommend_by_users(1)
it = recommend_by_items(1)

seen = set(info.loc[info["movieId"].isin(ratings.loc[ratings["userId"] == 1, "movieId"]), "title"])
print("1) Не советуем уже просмотренное:", not (set(u["Фильм"]) & seen))
print("2) У всех кандидатов >= 5 оценок:", (u["Оценок"] >= 5).all() and (it["Оценок"] >= 5).all())
print("3) Прогноз в шкале 0.5-5.0:", u["Прогноз"].between(0.5, 5).all() and it["Прогноз"].between(0.5, 5).all())
print("4) Разные значения прогноза (не везде 5.00):", u["Прогноз"].nunique(), "у зрителей,", it["Прогноз"].nunique(), "у фильмов")

1) Не советуем уже просмотренное: True
2) У всех кандидатов >= 5 оценок: True
3) Прогноз в шкале 0.5-5.0: True
4) Разные значения прогноза (не везде 5.00): 9 у зрителей, 5 у фильмов
